### Task 10: 

Create a mapping of pixel coordinates (that are fitted into a unit circle) to an arc of a circle with radius Rf, centred at the base of the object (the red star).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
from numpy import pi
import os
from PIL import Image

def load_img(image_path):
    try:
        img = Image.open(image_path) 
        return np.array(img)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None
    
def open_img():
    image_path = "stinkbug.png"  # replace this image file if needed
    if not os.path.exists(image_path): # error handling
        print(f"'{image_path}' not found")
        raise SystemExit
    else:
        img_path = load_img(image_path)
        if img_path is None:
            print('uhoh')
            raise SystemExit
        print(f"{image_path} successfully loaded")
        return img_path
    
def polar2cart(to_transform, start_theta, end_theta,h,w):
    flipped = np.flip(to_transform, (0,1)) # flip along x and y axes
    size = w *2  
    cartesian_img = np.zeros((size, size)+flipped.shape[2:],
                            dtype = flipped.dtype)
    max_r = h # bounding radius
    mask = np.ones((size, size), dtype=bool) # outside bounding circle = transparent
    center = size / 2
    angular_range = end_theta - start_theta 
    for y in range(size):
        for x in range(size):
            # convert coordinates to achieve anamorphic effect
            dx = (x - center)
            dy = y - center
            r = (dx**2 + dy**2)**0.5 - 100
            # check if r is within bounding r
            if r <= max_r and r >= 0:
                theta = np.atan2(dy, dx)
                # check if theta is within angular range
                if start_theta <= theta <= end_theta:
                    mask[y,x] = False
                    # mapping theta to column idx (normalise)
                    theta_normalized = (theta - start_theta) / angular_range
                    theta_idx = int(theta_normalized * w) % w
                    r_idx = int(r)
                    if r_idx < h:
                        cartesian_img[y, x] = flipped[r_idx, theta_idx]
    
    if len(cartesian_img.shape) == 3:
        masked_img = np.ma.array(cartesian_img, mask=np.stack([mask]*cartesian_img.shape[2], axis=2))
        rotated= np.rot90(masked_img, 3)
    else:
        masked_img = np.ma.array(cartesian_img, mask=mask)
        rotated= np.rot90(masked_img, 3)
    return rotated

def orig_data(orig_img):
    # creating the scaled meshgrid of the orig_img
    height, width = orig_img.shape[:2]
    aspect_ratio = height / width
    x0, y0 = 0 , 0    # object centre
    r = 1 # object stays within this radius
    obj_w = ( (r*2) / (width**2+height**2)**0.5 ) * width # this scales the image
    # minimum y coord
    y_min = y0 - obj_w / 2 * aspect_ratio
    # meshgrid
    x = np.linspace(0, obj_w, width) - obj_w/2 + x0
    y = -np.linspace(0, obj_w*aspect_ratio, height) + obj_w*aspect_ratio/2 + y0
    X, Y = np.meshgrid(x,y)
    return X,Y,y_min,height,width

def transform_data(trans_img,y_min):
    # creating the scaled meshgrid of the transformed_img
    height, width = trans_img.shape[:2]
    aspect_ratio = height / width
    x0,y0 = 0, y_min
    obj_w = 5 *2 # scales the image
    # meshgrid
    x = np.linspace(0, obj_w, width) - obj_w/2 + x0
    y = -np.linspace(0, obj_w*aspect_ratio, height) + obj_w*aspect_ratio/2 + y0
    X, Y = np.meshgrid(x,y)  # Non-uniform spacing allowed
    return X,Y

# opening image file and getting meshgrid for orig_img
orig_img = open_img()
X,Y,y_min,height,width = orig_data(orig_img)
# calculating and getting meshgrid for transformed image
start_theta = -3*pi/8
end_theta = 3*pi/8 # start off with total 135* of polar warp
trans_img = polar2cart(orig_img, start_theta, end_theta,height,width)
xx,yy = transform_data(trans_img,y_min)
# fig and axis
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
# cylinder outline and marker for centre of mapping
cylinder = Circle((0, 0),1 , edgecolor='b',
                      fc = 'None', lw=0.5, zorder = 5, label = 'Cylinder Outline')
ax.add_patch(cylinder)
marker = plt.plot(0,y_min, marker='*', color='red',
         markersize=3, zorder = 5, label='Mapping Centre')
# displaying the actual and transformed image
img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
trans_show = ax.pcolormesh(xx, yy, trans_img, shading='auto', zorder=3)
# axis position and limits
ax.spines['left'].set_position(('outward', 0.8)) # position of y-axis
ax.spines['bottom'].set_position(('outward', 0.8)) # position of x-axis
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 5)
# axes label
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
# title and text
ax.set_title('Anamorphic Transformation', fontsize=16)
plt.text(6.1,4.8, 'Up Arrow Key = Increase Angle', fontsize = 8)
plt.text(6.1,4.5, 'Down Arrow Key = Decrease Angle', fontsize = 8)
obj_txt = ax.text(0, 1.15, 'Actual Object', 
            ha='center', fontsize=10)
reflect_txt = ax.text(0, -5.75, 'Transformed Object', 
            ha='center', fontsize=10)
uhoh_txt = ax.text(6.1,4.1, '', color = 'red', fontsize = 8)
# grid
ax.grid(True, alpha=0.5, linestyle='-', zorder = 1)
ax.set_aspect('equal', adjustable='box')
#legend
plt.legend(loc='upper left', fontsize=10)

def update(event): # update transformation
    global start_theta, end_theta
    uhoh_txt.set_text('')
    obj_txt.set_text('')
    reflect_txt.set_text('')
    fig.canvas.draw_idle()
    # increasing angle
    if event.key == 'up':
        start_theta -= pi/12
        if start_theta <-pi-0.3:
            uhoh_txt.set_text('Out of Bounds')
            start_theta += pi/12
        else:
            end_theta += pi/12
            move_pic()
    # decreasing angle
    elif event.key == 'down':
        end_theta -= pi/12
        if end_theta <= 0:
            uhoh_txt.set_text('Out of Bounds')
            end_theta += pi/12
        else:
            start_theta += pi/12
            move_pic()

def move_pic():
    # actually transform image
    global trans_show
    trans_img = polar2cart(orig_img, start_theta, end_theta,height,width) # re-calc reflected img
    trans_show.remove()
    xx,yy = transform_data(trans_img,y_min)
    trans_show = ax.pcolormesh(xx, yy, trans_img, shading='auto', zorder=3)
    plt.show()

fig.canvas.mpl_connect('key_press_event', update)
plt.show()